In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely.geometry import Polygon
from shapely.ops import unary_union
import numpy as np
from datetime import datetime
import os


In [2]:
# Load IBTrACS shapefiles
points_gdf = gpd.read_file("IBTrACS.WP.list.v04r01.points.shp")
lines_gdf = gpd.read_file("IBTrACS.WP.list.v04r01.lines.shp")

# Load Philippines administrative boundary
boundary_gdf = gpd.read_file("PHL_adm0.shp")
buffer_gdf = gpd.read_file("PHL_1degree_buffer.shp")

In [3]:
# Filter for Typhoon Rai (2021) using SID 2021346N05145
rai_points = points_gdf[points_gdf['SID'] == '2021346N05145']
rai_lines = lines_gdf[lines_gdf['SID'] == '2021346N05145']

In [4]:
# Ensure CRS consistency
phl_buffer = buffer_gdf.to_crs("EPSG:4326")
phl_adm0 = boundary_gdf.to_crs("EPSG:4326")
rai_points = rai_points.to_crs("EPSG:4326")
rai_lines = rai_lines.to_crs("EPSG:4326")

# Convert ISO_TIME to datetime
rai_points['ISO_TIME'] = pd.to_datetime(rai_points['ISO_TIME'])
rai_lines['ISO_TIME'] = pd.to_datetime(rai_lines['ISO_TIME'])


In [5]:
rai_within_buffer = gpd.sjoin(rai_points, buffer_gdf, how='inner', predicate='within')
entry_time = rai_within_buffer['ISO_TIME'].min()
exit_time = rai_within_buffer['ISO_TIME'].max()


In [6]:
# Filter points within the time range
rai_impact_points = rai_points[(rai_points['ISO_TIME'] >= entry_time) &
                              (rai_points['ISO_TIME'] <= exit_time)]

rai_impact_lines = rai_lines[(rai_lines['ISO_TIME'] >= entry_time) &
                              (rai_lines['ISO_TIME'] <= exit_time)]


In [7]:
# Extract wind radius fields (USA_R64_NE/SE/SW/NW)
wind_fields = ['USA_R64_NE', 'USA_R64_SE', 'USA_R64_SW', 'USA_R64_NW']
impact_grid = []


In [9]:
# For each impact point, create a grid based on wind radii
for idx, point in rai_impact_points.iterrows():
    lon, lat = point.geometry.x, point.geometry.y
    time = point['ISO_TIME']

    # Get wind radii (in nautical miles)
    radii = {field: point[field] for field in wind_fields if pd.notnull(point[field])}

    # If radii exist, approximate the impact area
    if radii:
        # Convert nautical miles to degrees (approximate, 1 nm ≈ 1/60 degree)
        nm_to_deg = 1/60
        for quadrant, radius in radii.items():
            if radius > 0:
                # Calculate offsets based on quadrant
                if 'NE' in quadrant:
                    lon_offset, lat_offset = radius * nm_to_deg, radius * nm_to_deg
                elif 'SE' in quadrant:
                    lon_offset, lat_offset = radius * nm_to_deg, -radius * nm_to_deg
                elif 'SW' in quadrant:
                    lon_offset, lat_offset = -radius * nm_to_deg, -radius * nm_to_deg
                elif 'NW' in quadrant:
                    lon_offset, lat_offset = -radius * nm_to_deg, radius * nm_to_deg

                # Create grid point
                grid_point = {
                    'longitude': round(lon + lon_offset, 14),
                    'latitude': round(lat + lat_offset, 15),
                    'time': time,
                    'quadrant': quadrant,
                    'wind_radius_nm': radius
                }
                impact_grid.append(grid_point)

# Convert to DataFrame
impact_grid_df = pd.DataFrame(impact_grid)

# Save to CSV
if not impact_grid_df.empty:
    impact_grid_df.to_csv('rai_impact_grid.csv', index=False)
    print(f"Impact grid saved to 'rai_impact_grid.csv' with {len(impact_grid_df)} points.")
else:
    print("No impact grid points found within the buffer zone.")

Impact grid saved to 'rai_impact_grid.csv' with 52 points.
